In [24]:
from dotenv import load_dotenv
from dataclasses import dataclass
from langchain_community.utilities import SQLDatabase
from langchain_community.utilities import SQLDatabase
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.runtime import get_runtime
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI 


In [25]:
load_dotenv()

True

In [26]:

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [27]:
@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [28]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- Call execute_sql as many times as needed to answer the question.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 50 rows of output unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [29]:
#llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


agent = create_agent(
    model= ChatOllama(
    model="qwen3.5:9b",
    temperature=0,
    num_ctx=8192,
    num_predict=-1,
    reasoning=False, #Needed this to output the full text, as the <thinking> tokens stopped the final ouput to be full printed in the notebook
),
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,

)

In [30]:
question = "Which table has the largest number of entries?"
for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",

):
    step["messages"][-1].pretty_print()
    
    



================================ Human Message =================================

Which table has the largest number of entries?
================================== Ai Message ==================================

I need to find which table has the most rows. Let me first check what tables exist in the database, then count the rows in each table.
Tool Calls:
  execute_sql (455b2d71-589e-46a0-976a-2aaa0a9927f4)
 Call ID: 455b2d71-589e-46a0-976a-2aaa0a9927f4
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================

Now I'll count the number of rows in each table to determine which one has the largest number of entries.
Tool Cal

In [31]:
# Run this in a separate cell to inspect the raw message
steps = list(agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
))

last_msg = steps[-1]["messages"][-1]
print("content:", repr(last_msg.content))
print("reasoning_content:", repr(last_msg.additional_kwargs.get("reasoning_content", "")[:200]))
print("tool_calls:", last_msg.tool_calls)

content: "Based on the row counts, **`PlaylistTrack`** has the largest number of entries with **8,715 rows**.\n\nHere's the ranking of all tables by size:\n1. **PlaylistTrack**: 8,715 rows\n2. Track: 3,503 rows\n3. InvoiceLine: 2,240 rows\n4. Invoice: 412 rows\n5. Album: 347 rows\n6. Artist: 275 rows\n7. Customer: 59 rows\n8. Genre: 25 rows\n9. Playlist: 18 rows\n10. Employee: 8 rows\n11. MediaType: 5 rows"
reasoning_content: ''
tool_calls: []
